# Serve Qwen3-8B on a Colab A100 (remote inference for SWE-agent)

This notebook runs **only the neural-network inference**. It serves `Qwen/Qwen3-8B`
with vLLM behind an OpenAI-compatible API and exposes it through a public
Cloudflare tunnel.

On your **local machine (WSL2)** you then run SWE-agent + Docker and point it at
the printed URL:

```bash
export MODEL_API_BASE="https://<printed-tunnel>.trycloudflare.com/v1"
bash scripts/run_pilot_batch.sh config/pilot_instances.txt
```

**Runtime:** set **Runtime → Change runtime type → A100 GPU** before running.

> The same A100 is also where you run the GPU **projection** step
> (`stage2.extract.project_steps`) after ingesting trajectories — that step needs
> raw residual-stream activations and cannot go through this API.

In [ ]:
# 0. Confirm we actually have a GPU (expect A100 40GB).
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 1. Install vLLM + the Cloudflare tunnel client.
#    (Pin vLLM if you need to match a specific transformers/chat-template version.)
!pip -q install "vllm>=0.6.2"
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

In [ ]:
# 2. Config. enable_thinking=False MUST match the local replay/projection step
#    (stage2 defaults use enable_thinking: false), or activations won't line up.
MODEL = "Qwen/Qwen3-8B"
SERVED_NAME = "Qwen3-8B"   # must equal the suffix in MODEL_NAME (hosted_vllm/Qwen3-8B)
PORT = 8000
MAX_MODEL_LEN = 32768
API_KEY = "EMPTY"          # set a real token here AND in MODEL_API_KEY locally to lock down the tunnel

In [ ]:
# 3. Launch vLLM in the background and stream its log.
import subprocess, os, time, itertools

log = open("vllm.log", "w")
cmd = [
    "vllm", "serve", MODEL,
    "--served-model-name", SERVED_NAME,
    "--dtype", "bfloat16",
    "--max-model-len", str(MAX_MODEL_LEN),
    "--port", str(PORT),
    "--chat-template-kwargs", '{"enable_thinking": false}',
]
if API_KEY and API_KEY != "EMPTY":
    cmd += ["--api-key", API_KEY]
print(" ".join(cmd))
vllm_proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
print("vLLM starting (pid %d); first run downloads ~16GB of weights..." % vllm_proc.pid)

In [ ]:
# 4. Wait until the server answers /v1/models (model load can take a few minutes).
import requests, time

headers = {} if API_KEY in ("", "EMPTY") else {"Authorization": f"Bearer {API_KEY}"}
url = f"http://localhost:{PORT}/v1/models"
for _ in range(120):  # up to ~10 min
    if vllm_proc.poll() is not None:
        raise RuntimeError("vLLM exited early — see tail below:\n" + open("vllm.log").read()[-3000:])
    try:
        r = requests.get(url, headers=headers, timeout=5)
        if r.ok:
            print("vLLM is up:", r.json())
            break
    except Exception:
        pass
    time.sleep(5)
else:
    raise TimeoutError("vLLM did not become ready; check vllm.log")

In [ ]:
# 5. Open a public tunnel to the local vLLM port and print the URL to use locally.
import subprocess, re, time

tunnel_log = open("cloudflared.log", "w")
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://localhost:{PORT}", "--no-autoupdate"],
    stdout=tunnel_log, stderr=subprocess.STDOUT,
)
public = None
for _ in range(60):
    time.sleep(2)
    txt = open("cloudflared.log").read()
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", txt)
    if m:
        public = m.group(0)
        break
if not public:
    raise RuntimeError("No tunnel URL yet; check cloudflared.log")

print("\n" + "=" * 70)
print("Remote model endpoint is LIVE. On your local machine (WSL2) run:\n")
print(f'  export MODEL_API_BASE="{public}/v1"')
if API_KEY and API_KEY != "EMPTY":
    print(f'  export MODEL_API_KEY="{API_KEY}"')
print('  export MODEL_NAME="hosted_vllm/Qwen3-8B"')
print("  bash scripts/run_pilot_batch.sh config/pilot_instances.txt")
print("=" * 70)

In [ ]:
# 6. Smoke-test the public endpoint end-to-end (same path SWE-agent will use).
import requests
r = requests.post(
    f"{public}/v1/chat/completions",
    headers=headers,
    json={
        "model": SERVED_NAME,
        "messages": [{"role": "user", "content": "Reply with exactly: OK"}],
        "max_tokens": 8,
        "temperature": 0.0,
    },
    timeout=60,
)
print(r.status_code, r.json()["choices"][0]["message"]["content"])

In [ ]:
# 7. Keep this cell running to hold the tunnel + server open during the batch.
#    Interrupt it (or close the runtime) to shut everything down.
import time
try:
    while True:
        if vllm_proc.poll() is not None:
            print("vLLM exited; tail of log:\n", open("vllm.log").read()[-2000:])
            break
        time.sleep(30)
except KeyboardInterrupt:
    print("Shutting down vLLM + tunnel...")
    tunnel.terminate()
    vllm_proc.terminate()